# Interpretability — Psychotherapy Dropout Prediction (SHAP)

This notebook loads the trained **XGBoost** model and processed data, then uses **SHAP** (SHapley Additive exPlanations) to explain **why** the model assigns dropout risk at both **population** and **individual** levels.

**Why interpretability is non‑negotiable in medical AI:** Clinicians and patients must understand **what evidence** drives a risk score before acting on it. Opaque “black box” scores can harm trust, obscure harmful biases, and undermine shared decision‑making. Transparent, faithful explanations support safer use as **decision support**—not autonomous diagnosis.

## Imports and project path

We import **pandas**, **numpy**, **matplotlib**, and **SHAP** helpers from `src/evaluate`, plus **`prepare_data`** so **test features match training** (same split and scaling). The project root is added to `sys.path` and we **`chdir` to the project root** so `reports/` paths in `evaluate.py` resolve next to your code, not under `notebooks/`.

In [ ]:
import os
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap

for _path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), ".."))):
    if os.path.isdir(os.path.join(_path, "src")) and _path not in sys.path:
        sys.path.insert(0, _path)
        break

if os.path.basename(os.path.normpath(os.getcwd())).lower() == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
else:
    PROJECT_ROOT = os.getcwd()

os.chdir(PROJECT_ROOT)
os.makedirs(os.path.join(PROJECT_ROOT, "reports"), exist_ok=True)

from src.evaluate import (
    compute_risk_score,
    compute_shap_values,
    plot_global_importance,
    plot_individual_explanation,
)
from src.model import prepare_data

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

FEATURE_LABELS = {
    "patient_id": "Patient ID",
    "phq9_score": "PHQ-9 Depression Score",
    "session_number": "Session Number",
    "session_frequency_per_month": "Sessions Per Month",
    "attendance_consistency": "Attendance Consistency",
    "gap_between_sessions_days": "Days Between Sessions",
    "mood_rating": "Mood Rating",
    "age": "Patient Age",
    "phq9_change_rate": "PHQ-9 Change Rate",
    "attended": "Session Attended",
    "gap_increasing": "Increasing Gap Pattern",
    "max_attendance_streak": "Max Attendance Streak",
}

print("PROJECT_ROOT:", PROJECT_ROOT)

## Load trained XGBoost model

**Clinical meaning:** The **same** model artifact used in your modeling notebook encodes learned patterns between engagement, symptoms, and **dropout**. Loading it here lets us attribute predictions to features with SHAP—**not** to retrain.

In [ ]:
model_path = os.path.join(PROJECT_ROOT, "models", "xgboost_model.pkl")
with open(model_path, "rb") as f:
    model = pickle.load(f)

print("Loaded model from:", model_path)
print("Model type:", type(model).__name__)

## Load processed data and build **scaled test** features (`X_test`)

**Clinical meaning:** SHAP must explain the **exact inputs the model saw at training time**. We reload `featured_data.csv`, run **`prepare_data`** (same stratified split and **StandardScaler** as training), and build **`X_test`** as a DataFrame with **technical** column names for prediction. We then create **`X_test_display`** with **plain‑English** labels for charts.

In [ ]:
csv_path = os.path.join(PROJECT_ROOT, "data", "processed", "featured_data.csv")
df = pd.read_csv(csv_path)
print("Loaded data shape:", df.shape)

tech_cols = [c for c in df.columns if c != "dropout"]
X_train, X_test_arr, y_train, y_test, _scaler = prepare_data(df)
print("After prepare_data — X_test shape:", X_test_arr.shape)

X_test = pd.DataFrame(X_test_arr, columns=tech_cols)
X_test_display = X_test.rename(columns=lambda c: FEATURE_LABELS.get(c, c))
print("X_test (technical columns) shape:", X_test.shape)
print("X_test_display (plain English) shape:", X_test_display.shape)

## Compute SHAP values for the full test set

**Clinical meaning:** SHAP values approximate **fair credit assignment**: each feature’s contribution to the **difference** between this patient’s predicted dropout risk and the **average** prediction. Summing SHAP values (plus baseline) recovers the model output for tree models like XGBoost.

In [ ]:
explainer, shap_values = compute_shap_values(model, X_test_display)
print("SHAP values tensor shape:", shap_values.values.shape)
print("Number of test patients (rows):", len(X_test_display))

## Global SHAP summary (population-level drivers)

**Clinical meaning:** The **summary plot** ranks features by **average absolute SHAP impact** across patients—showing which variables **most often** move predictions away from the baseline. This supports team discussion of **system-level** risk drivers (e.g., gaps, mood, PHQ‑9 trajectory).

In [ ]:
plot_global_importance(shap_values, X_test_display)
print("Saved:", os.path.join(PROJECT_ROOT, "reports", "global_importance.png"))

## Individual explanation — patient at index **0**

**Clinical meaning:** A **waterfall** plot walks from the **average** prediction to **this patient’s** prediction, showing each feature’s push **toward higher or lower** dropout risk. This is the core “**why this score**” view for supervision and case discussion.

In [ ]:
plot_individual_explanation(explainer, shap_values, X_test_display, patient_index=0)
print("Saved:", os.path.join(PROJECT_ROOT, "reports", "patient_0_explanation.png"))

## Individual explanation — patient at index **1**

**Clinical meaning:** Comparing **two** patients highlights how **different feature profiles** produce different risk levels—helping staff see that the model is not a single “rule” but **context‑dependent** reasoning (still an approximation, not causal truth).

In [ ]:
plot_individual_explanation(explainer, shap_values, X_test_display, patient_index=1)
print("Saved:", os.path.join(PROJECT_ROOT, "reports", "patient_1_explanation.png"))

## Risk score and tier for **three** example patients (Low / Moderate / High)

**Clinical meaning:** `compute_risk_score` maps **predicted dropout probability** to a **0–100** score and **Low / Moderate / High** tiers. We print **three** test patients whose **tiers differ** so you can see how **SHAP** and **numeric risk** align (when possible).

In [ ]:
tier_to_indices = {"Low": [], "Moderate": [], "High": []}
for i in range(len(X_test)):
    out = compute_risk_score(model, X_test.iloc[[i]])
    tier_to_indices[out["risk_tier"]].append(i)

def pick_first(lst):
    return lst[0] if lst else None

idx_low = pick_first(tier_to_indices["Low"])
idx_mod = pick_first(tier_to_indices["Moderate"])
idx_high = pick_first(tier_to_indices["High"])

proba = model.predict_proba(X_test)[:, 1]
if idx_low is None:
    idx_low = int(np.argmin(proba))
if idx_mod is None:
    idx_mod = int(np.argsort(proba)[len(proba) // 2])
if idx_high is None:
    idx_high = int(np.argmax(proba))

examples = [
    ("Example A — Low risk (or low probability)", idx_low),
    ("Example B — Moderate risk (or middle probability)", idx_mod),
    ("Example C — High risk (or high probability)", idx_high),
]

for label, idx in examples:
    r = compute_risk_score(model, X_test.iloc[[idx]])
    print("=" * 60)
    print(label)
    print("  Test row index:", idx)
    print("  Risk score (0–100):", r["risk_score"])
    print("  Risk tier:", r["risk_tier"])
print("=" * 60)

## SHAP dependence plot for the **most important** feature

**Clinical meaning:** A **dependence** plot shows how the model’s **SHAP value** for one feature changes **as that feature’s value** changes across patients—revealing **non‑linear** or **threshold** behavior. **Interaction** coloring (default in SHAP) can show whether another feature **modifies** the effect (interpret cautiously).

In [ ]:
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top_idx = int(np.argmax(mean_abs_shap))
top_feature_name = X_test_display.columns[top_idx]
print("Most important feature (mean |SHAP|):", top_feature_name)

plt.figure(figsize=(10, 6))
shap.plots.scatter(shap_values[:, top_feature_name], show=False)
plt.title(f"SHAP dependence — {top_feature_name}")
plt.xlabel(top_feature_name)
plt.ylabel("SHAP value (impact on dropout risk)")
plt.tight_layout()
plt.show()

## How to read these explanations as a therapist (plain English)

- **Global summary (beeswarm):** Each dot is a **patient**. The **horizontal axis** is **SHAP impact** on the model’s output (higher dropout risk vs. baseline). **Color** (when shown) is the **feature’s actual value** (e.g., redder = higher PHQ‑9). **Features sorted top-to-bottom** are those that **most often** change the score across patients.
- **Waterfall (one patient):** Start at **base value** (average model output). Each bar **adds or subtracts** risk based on **that patient’s** feature. The **end** is the model’s predicted score for that patient. **Right** = pushes risk **up**; **left** = pushes risk **down**.
- **Risk score / tier:** `compute_risk_score` maps **predicted dropout probability** to a **0–100** score and **Low / Moderate / High** bands. **Do not** treat these as diagnoses—use **alongside** clinical judgment and conversation.
- **Dependence plot:** Shows how **SHAP** for **one feature** varies as that feature’s **value** changes across patients. **Interaction** coloring (if present) suggests **another feature** may **change** how the first feature matters—use as **hypothesis**, not proof.

## Ethical considerations and limitations

- **This is decision support, not care:** The model is **not** a clinician; it cannot assess safety, consent, or context outside the dataset.
- **Bias and fairness:** Training data may **under‑represent** groups or settings; **errors may not be equal** across demographics. **SHAP explains the model**, not the **truth** about patients.
- **Correlation ≠ causation:** SHAP reflects **associations** learned by the model. **Do not** use SHAP alone to infer **causal** effects of treatment.
- **Privacy:** Treat **individual patient** plots as **sensitive**; share only under appropriate consent and governance.
- **Calibration:** **Risk scores** may be **mis‑calibrated** if the model was not tested in your environment—**validate** before operational use.